# Gene ↔ Mutation Relation-Wise Merge

Merges Gene–Mutation triples from EvoAGE;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [14]:
import os
import pandas as pd

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_MUTATION/ALL_GENE_MUTATION.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Load KG Sources

### 1a. hald

In [15]:
hald = pd.read_csv(PROC_DIR + 'hald/Gene_Mutation_HALD.csv')
hald.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
hald.columns = hald.columns.str.lower()
print(f"hald: {len(hald):,} rows | columns: {list(hald.columns)}")
hald['kg_type'] = 'Aging'
hald['kg_source'] = 'HALD_KG'
hald['species'] = 'Homo species'

hald.head(2)

hald: 274 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_id_is']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_id_is,kg_type,species
0,ADIPOQ,Gene_Mutation,rs2805533,Gene,associated,Mutation,HALD_KG,"adiponectin, C1Q and collagen domain containing",NCBI_ID,Aging,Homo species
1,CRP,Gene_Mutation,rs2619112,Gene,associated,Mutation,HALD_KG,C-reactive protein,NCBI_ID,Aging,Homo species


## 2. Check and Fix Duplicate Columns

In [16]:
SOURCE_DFS = [('hald', hald)]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[hald] ✓ no duplicates


## 3. Align Schemas and Concatenate

In [17]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 274 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,ADIPOQ,Gene_Mutation,rs2805533,Gene,associated,Mutation,HALD_KG,NCBI_ID,<NA>,"adiponectin, C1Q and collagen domain containing",<NA>,Aging,Homo species
1,CRP,Gene_Mutation,rs2619112,Gene,associated,Mutation,HALD_KG,NCBI_ID,<NA>,C-reactive protein,<NA>,Aging,Homo species


## 4. Standardise Fixed-Value Columns

In [18]:
final_df['head_type'] = 'Gene'
final_df['tail_type'] = 'Mutation'
final_df['relation']  = 'Gene_Mutation'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Gene_Mutation'}
Unique head_id_is: {'NCBI_ID'}
Unique tail_id_is: {<NA>}
Unique kg_source: {'HALD_KG'}


## 5. Deduplicate and Add Schema Columns

In [19]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"After deduplication: {len(final_df_dedup):,} rows")
final_df_dedup.head()

After deduplication: 252 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,ABCA1,Gene_Mutation,rs1061170,Gene,associated,Mutation,HALD_KG,NCBI_ID,None,ATP binding cassette subfamily A member 1,None,Aging,Homo species
1,ABCC2,Gene_Mutation,rs777902199,Gene,complicated,Mutation,HALD_KG,NCBI_ID,None,ATP binding cassette subfamily C member 2,None,Aging,Homo species
2,ABCC8,Gene_Mutation,rs2237982,Gene,associated,Mutation,HALD_KG,NCBI_ID,None,ATP binding cassette subfamily C member 8,None,Aging,Homo species
3,ABCC8,Gene_Mutation,rs7105832,Gene,associated,Mutation,HALD_KG,NCBI_ID,None,ATP binding cassette subfamily C member 8,None,Aging,Homo species
4,ABCC9,Gene_Mutation,rs704180,Gene,associated,Mutation,HALD_KG,NCBI_ID,None,ATP binding cassette subfamily C member 9,None,Aging,Homo species


## 6. QC — NaN Check

In [20]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,252,0,252
head_detail_name,0,0,0


## 7. Save Output

In [21]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 252 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_MUTATION/ALL_GENE_MUTATION.csv


In [22]:
# f = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_MUTATION/ALL_GENE_MUTATION.csv')
# f

In [23]:
set(final_df_dedup['kg_type'])

{'Aging'}